Load Packages

In [1]:
import pandas as pd
import geopandas as gpd
import folium
from shapely.geometry import LineString, Point
from bus_route_plot import original_route, new_route, original_and_new_route


Data Pre-Processing (Weekdays, Peak Hour)

In [3]:
df_202407 = pd.read_csv("Passenger_Volume_By_Bus_Stop/transport_node_bus_202407.csv")
df_202408 = pd.read_csv("Passenger_Volume_By_Bus_Stop/transport_node_bus_202408.csv")
df_202409 = pd.read_csv("Passenger_Volume_By_Bus_Stop/transport_node_bus_202409.csv")
combined_df = pd.concat([df_202407, df_202408, df_202409], ignore_index=True)

# Filter for DAY_TYPE == 'WEEKDAY' and TIME_PER_HOUR for peak hours [7, 8, 9, 10, 17, 18, 19, 20]
filtered_df = combined_df[(combined_df['DAY_TYPE'] == 'WEEKENDS/HOLIDAY') & 
                          (combined_df['TIME_PER_HOUR'].isin([9, 10, 11, 12, 17, 18, 19, 20, 21]))]
summarised_df = pd.DataFrame(columns=['PT_CODE', 'TOTAL_VOLUME'])
# Group by 'PT_CODE' and calculate the sum of tap-in and tap-out volumes
grouped = filtered_df.groupby('PT_CODE').agg(
    TOTAL_TAP_IN_VOLUME=('TOTAL_TAP_IN_VOLUME', 'sum'),
    TOTAL_TAP_OUT_VOLUME=('TOTAL_TAP_OUT_VOLUME', 'sum')).reset_index()
# Create a new column for the total volume (sum of tap-in and tap-out volumes)
grouped['TOTAL_VOLUME'] = grouped['TOTAL_TAP_IN_VOLUME'] + grouped['TOTAL_TAP_OUT_VOLUME']
summarised_df['PT_CODE'] = grouped['PT_CODE']
summarised_df['TOTAL_VOLUME'] = grouped['TOTAL_VOLUME']
summarised_df = summarised_df.sort_values(by='TOTAL_VOLUME', ascending=False)
trunkroutes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")
trunkroutes_grouped = trunkroutes.groupby('BusStopCode').first().reset_index()
# Merge summarised_df with trunkroutes based on PT_CODE == BusStopCode
merged_df = pd.merge(summarised_df, trunkroutes_grouped[['BusStopCode', 'Description', 'Latitude', 'Longitude','Direction']], 
                     left_on='PT_CODE', right_on='BusStopCode', how='left')
merged_df = merged_df.drop(columns=['BusStopCode'])
merged_df.to_csv('location_popular_bus_Stops.csv', index=False)
filtered_df = merged_df[~merged_df['Description'].str.contains('Stn|Int', na=False)].reset_index().drop('index', axis=1)

Retrieving MRT Map

In [4]:
from mrt_map import get_mrt_map

singapore = get_mrt_map()
singapore

/Users/lilyrozana/Documents/GitHub/DSA4264/venv/lib/python3.11/site-packages/pyogrio/raw.py:198: RuntimeWarning: TrainStation_Jul2024/repaired_shapefile.shp contains polygon(s) with rings with invalid winding order. Autocorrecting them, but that shapefile should be corrected using ogr2ogr for example.
  return ogr_read(
/Users/lilyrozana/Documents/GitHub/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geom.x)).tolist()
/Users/lilyrozana/Documents/GitHub/DSA4264/mrt_map.py:71: UserWarning: Geometry is in a geographic CRS. Results from 'centroid' are likely incorrect. Use 'GeoSeries.to_crs()' to re-project geometries to a projected CRS before this operation.

  coordinates = group_sorted['geometry'].centroid.apply(lambda geom: (geom.y, geom.x)).tolist()


Plotting popular bus stops across SG on weekdays, during peak hour

In [5]:
def scale_marker_size(volume, min_size=5, max_size=15):
    volume_range = merged_df['TOTAL_VOLUME'].max() - merged_df['TOTAL_VOLUME'].min()
    if volume_range == 0:
        return min_size  # Avoid division by zero
    scaled_size = ((volume - merged_df['TOTAL_VOLUME'].min()) / volume_range) * (max_size - min_size) + min_size
    return scaled_size

# Plot first 100 rows on the singapore_mrt map
for idx, row in filtered_df.head(100).iterrows():
    lat, lon = row['Latitude'], row['Longitude']
    total_volume = row['TOTAL_VOLUME']
    pt_code = row['PT_CODE']
    direction = row['Direction']
    
    if not pd.isna(lat) and not pd.isna(lon):
        popup_html = f"""
        <div style="font-size: 12px;">
            <strong>Bus Stop {pt_code}</strong><br>
            Volume: {total_volume}<br>
            Latitude: {lat:.6f}, Longitude: {lon:.6f}
            Direction: {direction}
        </div>
        """

        folium.CircleMarker(
            location=[lat, lon],
            radius=scale_marker_size(total_volume), 
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6,
            popup=folium.Popup(popup_html, max_width=150)
        ).add_to(singapore)
singapore

Check if selected bus stops are currently in any bus service route

In [25]:
route1a= ['17041', '17159', '12091', '43181', '43419', '28461', '28491', '28511', '28401', '21441']
route1a_proposed = ['17171', '17041', '17159', '12091', '43181', '43419', 
                    '28461', '28491', '28511', '28401','21431', '22009']

route1b = ['63241', '63059', '64419', '64119', '64111', '66331', '66339', '66271', 
        '54481', '54489', '54248', '54241', '53389', '53381', '52059', '52361', 
        '60089', '60179', '60161', '80071', '80069', '80089']
route1b_proposed = ['64009', '63249', '63059', '64419', '64119', '66339', 
                    '54489', '54248', '53389', '52059', '52361', '60081', 
                    '60179', '80071', '80089', '80069', '80009']

In [27]:
trunk_bus_routes = pd.read_csv("Bus_RoutesStopsServices/trunkroutes.csv")

# Retrieving the bus stop code from the bus stop label
busstop1 = trunk_bus_routes[trunk_bus_routes['Description'].str.contains('Keming Pr Sch', case=False, na=False)]['BusStopCode'].unique()
busstop2 = trunk_bus_routes[trunk_bus_routes['Description'].str.contains('Aft Bt Batok Stn/Blk 628', case=False, na=False)]['BusStopCode'].unique()

# Function for checking proportion of bus stops in proposed routes that are present in any bus service
trunk_bus_routes['BusStopCode'] = trunk_bus_routes['BusStopCode'].astype(str)
def calculate_proportion(group, list_of_buses):
    matching_stops = group['BusStopCode'].isin(list_of_buses).sum()
    proportion = matching_stops / len(group)
    return proportion

# Check if there is an existing bus service that goes through the bus stops for route1a
grouped = trunk_bus_routes.groupby(['ServiceNo', 'Direction']).apply(lambda group: calculate_proportion(group, route1a)).reset_index(name='Proportion')
grouped_sorted = grouped.sort_values(by='Proportion', ascending=False)
grouped_sorted

# Plot that bus service on the map
# original_route(grouped_sorted['ServiceNo'].iloc[0])

# Check if there is an existing bus service that goes through the bus stops for route1b
grouped = trunk_bus_routes.groupby(['ServiceNo', 'Direction']).apply(lambda group: calculate_proportion(group, route1b)).reset_index(name='Proportion')
grouped_sorted = grouped.sort_values(by='Proportion', ascending=False)
grouped_sorted

/var/folders/06/3zsj3_kd48n_rbnyhcxkz01c0000gn/T/ipykernel_10641/2935348950.py:14: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  grouped = trunk_bus_routes.groupby(['ServiceNo', 'Direction']).apply(lambda group: calculate_proportion(group, route1a)).reset_index(name='Proportion')
/var/folders/06/3zsj3_kd48n_rbnyhcxkz01c0000gn/T/ipykernel_10641/2935348950.py:19: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  

,ServiceNo,Direction,Proportion
231,21A,1,0.250000
128,159B,1,0.200000
364,72B,1,0.200000
88,13A,1,0.166667
32,115,1,0.153846
...,...,...,...
488,97,1,0.000000
487,96B,1,0.000000
486,96A,1,0.000000
485,969A,1,0.000000


Plotting Proposed Bus Routes for Weekday Peak Hour Express Service

In [26]:
# Plotting Proposed Route 1a
filtered_stops = trunk_bus_routes[trunk_bus_routes['BusStopCode'].isin(route1a_proposed)]
filtered_stops['BusStopCode'] = filtered_stops['BusStopCode'].astype(str)  
filtered_stops = filtered_stops.set_index('BusStopCode').loc[route1a_proposed].reset_index()

for _, row in filtered_stops.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"Bus Stop: {row['BusStopCode']}, {row['Description']}",
        icon=folium.Icon(color="blue", icon="info-sign"),
    ).add_to(singapore)

bus_stop_coords = filtered_stops[['Latitude', 'Longitude']].values.tolist()  
folium.PolyLine(bus_stop_coords, color="black", weight=5, opacity=1).add_to(singapore)

# Plotting Proposed Route 1b
filtered_stops = trunk_bus_routes[trunk_bus_routes['BusStopCode'].isin(route1b_proposed)]
filtered_stops['BusStopCode'] = filtered_stops['BusStopCode'].astype(str)  
filtered_stops = filtered_stops.set_index('BusStopCode').loc[route1b_proposed].reset_index()

for _, row in filtered_stops.iterrows():
    folium.Marker(
        location=[row['Latitude'], row['Longitude']],
        popup=f"Bus Stop: {row['BusStopCode']}, {row['Description']}",
        icon=folium.Icon(color="blue", icon="info-sign"),
    ).add_to(singapore)

bus_stop_coords = filtered_stops[['Latitude', 'Longitude']].values.tolist()  
folium.PolyLine(bus_stop_coords, color="black", weight=5, opacity=1).add_to(singapore)

singapore

/var/folders/06/3zsj3_kd48n_rbnyhcxkz01c0000gn/T/ipykernel_10641/2720793556.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_stops['BusStopCode'] = filtered_stops['BusStopCode'].astype(str)
/var/folders/06/3zsj3_kd48n_rbnyhcxkz01c0000gn/T/ipykernel_10641/2720793556.py:16: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  filtered_stops['BusStopCode'] = filtered_stops['BusStopCode'].astype(str)
